# TP2 - Etapa 2 en Google Colab: dataset, preprocesamiento, entrenamiento y comparacion de modelos

Notebook de trabajo de la Etapa 2: analisis del dataset, preprocesamiento, fine-tuning de
ResNet18 (Modelo A, obligatorio), CNN propia (Modelo B, opcional) y estudio comparativo.

El entrenamiento se realiza en Colab para aprovechar la GPU:
**Entorno de ejecucion -> Cambiar tipo de entorno de ejecucion -> GPU (T4)**.

Requisitos previos (en tu fork):
- `train_classifier`, `evaluate_classifier` y `extract_custom_embedding` implementadas
  en `src/lib/services/classifier_service.py` (esta notebook solo las orquesta).

Al finalizar:
- Descargar los checkpoints generados y colocarlos en `models/` de tu entorno local
  (la aplicacion los usa en las pestañas Etapa 1 y 2 del frontend).
- Publicarlos en un link de solo lectura publico para los docentes.
- Entregar esta notebook ejecutada, con sus salidas.

## Equipo
- Alumno 1: Agustín Accurso
- Alumno 2: Lautaro Cena

## 1. Clonar el repositorio

Si tu fork es privado, genera un token de acceso (GitHub -> Settings -> Developer settings ->
Personal access tokens) y usalo en la URL, o sube un zip del proyecto a Colab/Drive.

In [1]:
# TO-DO: completar con la URL de tu fork.
# Repo privado: https://<TOKEN>@github.com/<usuario>/tuia-dog-recognition-app.git
REPO_URL = "https://github.com/accursoagus/tuia-dog-recognition-app.git"

!git clone $REPO_URL proyecto
%cd proyecto

c:\Users\Agus\Desktop\Computer Vision\TP 2\tuia-dog-recognition-app\proyecto


fatal: destination path 'proyecto' already exists and is not an empty directory.


## 2. Instalar dependencias

Colab ya incluye torch, torchvision, opencv, numpy, scikit-learn y matplotlib;
solo se instala lo que falta.

In [2]:
!pip install -q pydantic-settings python-dotenv kagglehub albumentations


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import sys
import subprocess
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets

from sklearn.metrics import confusion_matrix

## 3. Configuracion del entorno

En Colab no hay PostgreSQL; la Etapa 2 no usa la base vectorial, por lo que se desactiva
pgvector. La configuracion se define por variables de entorno **antes** de importar `lib`.

In [ ]:
os.environ["USE_PGVECTOR"] = "false"
from dotenv import load_dotenv
load_dotenv(ROOT / ".env.local.example")

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    # Si el notebook no corre desde la raiz del proyecto, se busca hacia arriba
    for parent in ROOT.parents:
        if (parent / "src").is_dir():
            ROOT = parent
            break

sys.path.insert(0, str(ROOT / "src"))
print("ROOT detectado:", ROOT)

from lib.config import settings
from lib.bootstrap import build_classifier

print("batch_size:", settings.batch_size, "| max_epochs:", settings.max_epochs)
print("dataset_path:", settings.dataset_path)
print("GPU disponible:", torch.cuda.is_available())

ROOT detectado: c:\Users\Agus\Desktop\Computer Vision\TP 2\tuia-dog-recognition-app
batch_size: 32 | max_epochs: 15
dataset_path: data\dataset
GPU disponible: False


## 4. Descargar el dataset

In [5]:
!python scripts/download_dataset.py
!ls data/dataset

python: can't open file 'c:\\Users\\Agus\\Desktop\\Computer Vision\\TP 2\\tuia-dog-recognition-app\\proyecto\\scripts\\download_dataset.py': [Errno 2] No such file or directory
"ls" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


## 5. Analisis del dataset

TO-DO:
- Analizar la distribucion de clases y documentar la cantidad de imagenes por raza.
- Definir los conjuntos de entrenamiento, validacion y prueba.
- Construir un conjunto independiente para evaluacion (ej: imagenes descargadas de internet).
- Filtrar imagenes de baja calidad (documentar el criterio).

In [8]:
import subprocess
result = subprocess.run(["python", "scripts/download_dataset.py"], cwd=ROOT, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
print("dataset existe ahora:", (settings.dataset_path / "train").is_dir())


kagglehub no esta instalado. Instala las dependencias (requirements.txt) o descarga el dataset manualmente desde Kaggle.

dataset existe ahora: False


In [7]:
print(settings.dataset_path)
print(settings.dataset_path.exists())
print(list(settings.dataset_path.parent.iterdir()) if settings.dataset_path.parent.exists() else "ni el padre existe")

data\dataset
False
ni el padre existe


In [6]:
breeds = sorted(p.name for p in (settings.dataset_path / "train").iterdir() if p.is_dir())
counts = {b: len(list((settings.dataset_path / "train" / b).glob("*.jpg"))) for b in breeds}

df_counts = pd.Series(counts).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(16, 5))
df_counts.plot(kind="bar", ax=ax, title="Imagenes por raza (train)")
ax.set_xticklabels(df_counts.index, rotation=90, fontsize=6)
plt.tight_layout()
plt.savefig("output/class_distribution_etapa2.png", dpi=100)
plt.show()

print(f"Total razas: {len(breeds)}")
print(f"Min/Max imagenes por raza: {df_counts.min()} / {df_counts.max()}")
print(f"Promedio de imagenes por raza: {df_counts.mean():.1f}")
print(f"Desvio estandar: {df_counts.std():.1f}")
print(f"Razas con menos de 100 imagenes: {(df_counts < 100).sum()} de {len(df_counts)}")

for split in ("train", "valid", "test"):
    split_path = settings.dataset_path / split
    n_images = sum(1 for _ in split_path.rglob("*.jpg"))
    n_breeds = sum(1 for p in split_path.iterdir() if p.is_dir())
    print(f"{split}: {n_images} imagenes, {n_breeds} razas")

FileNotFoundError: [WinError 3] El sistema no puede encontrar la ruta especificada: 'data\\dataset\\train'

**Conclusiones del analisis exploratorio (Etapa 2):**

El dataset presenta 70 razas. A diferencia de la Etapa 1, donde el
desbalance de clases no afecta por usar un modelo pre-entrenado sin
ajuste, en esta etapa sí es relevante: el fine-tuning aprende
directamente de la distribucion de clases de `train`, por lo que las
razas con menos imagenes son mas propensas a bajo recall. Esto se
retoma en la seccion de analisis de errores tras evaluar ambos
modelos.

## 6. Preprocesamiento

TO-DO: definir y justificar resize, normalizacion y data augmentation (horizontal flip,
rotacion, blur, variaciones de brillo y contraste, ruido).

In [ ]:
from lib.bootstrap import build_classifier

classifier_preview = build_classifier(settings)
train_tf = classifier_preview._build_transforms(train=True)
valid_tf = classifier_preview._build_transforms(train=False)

# Tomamos una imagen de ejemplo
sample_breed = breeds[0]
sample_path = next((settings.dataset_path / "train" / sample_breed).glob("*.*"))

from PIL import Image
img = Image.open(sample_path).convert("RGB")

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
axes[0].imshow(img)
axes[0].set_title("Original")
for i in range(1, 5):
    augmented = train_tf(img)
    # des-normalizar para visualizar
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1) if 'IMAGENET_MEAN' in dir() else torch.tensor([0.485,0.456,0.406]).view(3,1,1)
    std = torch.tensor(IMAGENET_STD).view(3,1,1) if 'IMAGENET_STD' in dir() else torch.tensor([0.229,0.224,0.225]).view(3,1,1)
    denorm = (augmented * std + mean).clamp(0,1).permute(1,2,0).numpy()
    axes[i].imshow(denorm)
    axes[i].set_title(f"Augmentation {i}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig("output/augmentation_examples.png", dpi=100)
plt.show()

## 7. Modelo A (obligatorio): fine-tuning de ResNet18

`train_classifier` debe guardar el checkpoint en `models/resnet18_finetuned.pth`
(o el nombre configurado en RESNET18_MODEL_NAME). Documentar los hiperparametros
utilizados (learning rate, batch size, epochs, optimizador, scheduler).

In [ ]:
from lib.bootstrap import build_classifier

classifier = build_classifier(settings)
classifier.set_active_model("resnet18_finetuned")
classifier.train_classifier()

In [ ]:
metrics_a = classifier.evaluate_classifier()
metrics_a

### Curvas de entrenamiento y matriz de confusion (Modelo A)

TO-DO: graficar las curvas de entrenamiento/validacion y la matriz de confusion.

In [ ]:
## TO-DO: curvas de entrenamiento y matriz de confusion del Modelo A.

## 8. Modelo B (opcional, recomendado): CNN propia

`train_classifier` con el modelo activo `cnn_custom` debe guardar el checkpoint en
`models/cnn_custom.pth` (o el nombre configurado en CNN_CUSTOM_MODEL_NAME).

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

checkpoint_a = torch.load(settings.model_path / settings.resnet18_model_name, map_location="cpu", weights_only=False)
history_a = checkpoint_a["history"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history_a["train_loss"], label="train")
axes[0].plot(history_a["valid_loss"], label="valid")
axes[0].set_title("Loss - Modelo A")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history_a["train_acc"], label="train")
axes[1].plot(history_a["valid_acc"], label="valid")
axes[1].set_title("Accuracy - Modelo A")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.savefig("output/curves_model_a.png", dpi=100)
plt.show()

# Matriz de confusion sobre test
class_names = checkpoint_a["class_names"]
model_a = classifier._build_resnet18_finetuned(len(class_names))
model_a.load_state_dict(checkpoint_a["model_state_dict"])
model_a.eval()

test_tf = classifier._build_transforms(train=False)
test_ds = datasets.ImageFolder(settings.dataset_path / "test", transform=test_tf)
test_loader = DataLoader(test_ds, batch_size=settings.batch_size, shuffle=False)

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(classifier.device)
        outputs = model_a(images)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(16, 14))
sns.heatmap(cm, xticklabels=class_names, yticklabels=class_names, cmap="Blues")
plt.title("Matriz de confusion - Modelo A")
plt.xticks(rotation=90, fontsize=6)
plt.yticks(rotation=0, fontsize=6)
plt.tight_layout()
plt.savefig("output/confusion_matrix_a.png", dpi=100)
plt.show()

## 9. Estudio comparativo

TO-DO (pequeño estudio de la Etapa 2):
- Comparar los modelos entrenados: accuracy, precision, recall, specificity, F1.
- Analizar las clases con peor desempeño (falsos positivos / falsos negativos).
- Discutir trade-offs (performance vs costo computacional).
- Copiar las conclusiones al informe (`informe.ipynb`).

In [ ]:
classifier.set_active_model("cnn_custom")
classifier.train_classifier()
metrics_b = classifier.evaluate_classifier()
metrics_b

In [ ]:
import pandas as pd

comparison = pd.DataFrame([metrics_a, metrics_b], index=["ResNet18 Fine-Tuned", "CNN Custom"])
comparison

## 10. Descargar los checkpoints

Descargalos y colocalos en `models/` de tu entorno local para que la aplicacion los use
(pestañas Etapa 1 y 2 del frontend). Recorda publicarlos en un link de solo lectura publico.

In [ ]:
!ls -lh models/

try:
    from google.colab import files

    files.download(str(settings.model_path / settings.resnet18_model_name))
    # files.download(str(settings.model_path / settings.cnn_custom_model_name))
except ImportError:
    print("Fuera de Colab: los checkpoints quedan en models/")